# 08 自动微分

本节学习 PyTorch 自动微分的最小用法：`requires_grad`、计算图、`backward()`、`.grad`、梯度清零。

## 1. 最小概念

- `requires_grad=True`：告诉 PyTorch 需要跟踪这个张量的计算过程。
- 计算图：PyTorch 会记录张量是如何一步步算出来的。
- `backward()`：从最终结果开始，沿计算图反向计算梯度。
- `.grad`：保存某个张量对应的梯度。
- 梯度清零：梯度默认会累加，下一次反向传播前通常要把梯度清零。

In [ ]:
import torch

torch.set_printoptions(precision=4)


def show(name, value):
    print(f"{name}: {value}")
    print(f"  shape: {tuple(value.shape)}")
    print(f"  requires_grad: {value.requires_grad}")
    print(f"  grad_fn: {value.grad_fn}\n")

## 2. 两层计算小例子

我们构造一个两层计算：

第一层：`hidden = x * w + b`

第二层：`prediction = hidden ** 2`

损失：`loss = (prediction - target) ** 2`

这里 `w` 和 `b` 是要学习的参数，所以设置 `requires_grad=True`。

In [ ]:
x = torch.tensor(2.0)
target = torch.tensor(25.0)

w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

hidden = x * w + b
prediction = hidden ** 2
loss = (prediction - target) ** 2

show("x", x)
show("w", w)
show("b", b)
show("hidden = x * w + b", hidden)
show("prediction = hidden ** 2", prediction)
show("loss = (prediction - target) ** 2", loss)

## 3. backward 和 grad

调用 `loss.backward()` 后，PyTorch 会从 `loss` 开始反向传播，把 `loss` 对 `w`、`b` 的梯度存到 `w.grad` 和 `b.grad`。

In [ ]:
loss.backward()

print("loss:", loss.item())
print("w.grad = dloss/dw:", w.grad)
print("b.grad = dloss/db:", b.grad)

## 4. 用梯度更新参数

梯度表示当前参数让损失变大的方向。想让损失变小，就沿梯度的反方向更新参数：

`参数 = 参数 - 学习率 * 梯度`

In [ ]:
lr = 0.001

with torch.no_grad():
    new_w = w - lr * w.grad
    new_b = b - lr * b.grad
    new_hidden = x * new_w + new_b
    new_prediction = new_hidden ** 2
    new_loss = (new_prediction - target) ** 2

print("更新前 w, b:", w.item(), b.item())
print("更新后 new_w, new_b:", new_w.item(), new_b.item())
print("更新前 loss:", loss.item())
print("更新后 new_loss:", new_loss.item())

## 5. 梯度清零

PyTorch 的梯度会累加。如果连续调用 `backward()`，不清零的话，新的梯度会加到旧梯度上。

训练循环里常见写法是：

`optimizer.zero_grad()` 或手动把 `.grad` 设为 `None`。

In [ ]:
print("清零前 w.grad:", w.grad)
print("清零前 b.grad:", b.grad)

w.grad = None
b.grad = None

print("\n清零后 w.grad:", w.grad)
print("清零后 b.grad:", b.grad)

second_hidden = x * w + b
second_prediction = second_hidden ** 2
second_loss = (second_prediction - target) ** 2
second_loss.backward()

print("\n第二次 backward 后 w.grad:", w.grad)
print("第二次 backward 后 b.grad:", b.grad)

## 小结

- 需要求梯度的参数要设置 `requires_grad=True`。
- 中间结果会带着 `grad_fn`，说明它来自计算图。
- `backward()` 会把梯度写入叶子张量的 `.grad`。
- 梯度会累加，下一轮计算前要清零。